In [35]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Estimator
import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import Statevector, random_clifford

from tomography import Random_Measurements
from tomography import NearSparseTomography, NearSparseTomography_v2 

from time import time 

In [36]:
NQs = [ 3 ]
shots = 1000
simulator_ideal=Estimator(backend_options={'shots':shots,
                                            'method':"stabilizer",},
                            transpile_options={'optimization_level':0},
                            abelian_grouping=True, ) 

In [37]:
MC = 10
Fids = np.zeros( [2,len(NQs),MC] )
num_meas = [ 40 ]

t1 = time() 
for i, NQ in enumerate(NQs):
    
    d = 2**NQ

    for j in range(MC): 
        RM = Random_Measurements(NQ)

        psi_circ = random_clifford( NQ ).to_circuit()
        psi_th = np.array( Statevector(psi_circ) )

        RM.RandomMeasurements( num_meas[i], psi_circ, 
                                simulator_ideal )

        phi_in = np.random.rand(d) + 1j * np.random.rand(d)
        phi_in = phi_in / np.linalg.norm(phi_in)

        psi_out = NearSparseTomography( phi_in, RM ) 
        fid = np.abs( np.vdot( psi_th, psi_out) )**2  
        Fids[0,i,j] = fid
        psi_out1 = psi_out

        psi_out = NearSparseTomography_v2( psi_out, RM ) 
        fid = np.abs( np.vdot( psi_th, psi_out@psi_th ) )  
        Fids[1,i,j] = fid
t2 = time() 
t2-t1 

6.417223691940308

In [38]:
import cvxpy as cp 

In [ ]:
# def LocalProduct_cvx( Psi, Operators , Dims=[] ):
#     """
#     Calculate the product (A1xA2x...xAn)|psi>
#     """
#     shape = Psi.shape
#     if len(shape) > 1 :
#         num_vecs = shape[1]
#     else:
#         num_vecs = 1
#     if not Dims: 
#         Dims = [ Operators[k].shape[-1] for k in range( len(Operators) ) ]
#     N = len(Dims)
#     Dim = np.prod(Dims)
#     for k in range(N):
#             Psi  = (( Operators[k]@cp.reshape(Psi,
#                                             (Dims[k],
#                                             num_vecs*Dim//Dims[k])) 
#                                             ).T )
#     return cp.reshape( Psi, (num_vecs,Dim) ).T

# def to_base(number, base, fill=None):
#     """Converts a non-negative number to a list of digits in the given base.
#     The base must be an integer greater than or equal to 2 and the first digit
#     in the list of digits is the most significant one.
#     """
#     if not number:
#         digits = [0]
#     else:
#         digits = []
#         while number:
#             digits.append(number % base)
#             number //= base
#     if fill:
#         for _ in range( fill-len(digits) ):
#             digits.append( 0 )
#     return list(reversed(digits))

# def exp_val_cvx( rho, j, MDF ):
#     ExpVal = cp.trace(LocalProduct_cvx(rho,
#                         [MDF.Sigmamu[k] 
#                         for k in to_base(j,4,MDF.NQ)]
#                         ))/np.sqrt(2**MDF.NQ) 
#     return ExpVal 

# def NearSparseTomography_v3(phi, MDF):

#     sigmas = []
#     expvals = []
#     for j in MDF.Measures:
#         expvals.append( MDF.Measures[j] )
#         sigma = 1
#         for k in to_base(j,4,MDF.NQ):
#             sigma = np.kron(sigma, MDF.Sigmamu[k])
#         sigmas.append( sigma )

#     sigmas = np.real( sigmas ).reshape(-1,4**MDF.NQ).conj()/np.sqrt(2**MDF.NQ)
#     expvals = np.array(expvals).reshape(-1,1)

#     dim = phi.shape[0]
#     rho = cp.Variable((dim, dim), hermitian=True)
#     constraints = [rho >> 0] 
#     # constraints += [ cp.trace(rho) == 1 ] 
#     constraints += [ cp.norm(sigmas@cp.reshape( rho, (4**MDF.NQ,1) )-expvals) <= 0.1 ]
#     # constraints += [ exp_val_cvx(rho,j,MDF)==MDF.Measures[j]
#     #                 for j in MDF.Measures ] 
#     objective = cp.Minimize( cp.normNuc(rho) )
#     problema = cp.Problem( objective, constraints )
#     problema.solve()
#     return rho.value #/cp.trace(rho.value)

In [ ]:
# rho = NearSparseTomography_v3( phi_in, RM ) 
# np.linalg.eig( rho )[0] , np.vdot( psi_th, rho@psi_th )